In [ ]:
import os
import json
import openai
from openai import OpenAI
import pandas as pd
import chardet

print(openai.__version__)

client = OpenAI()

1.99.9


In [13]:
data_path = "/Users/peterhoffman/Library/CloudStorage/Dropbox/common_experience_2025/contracts/"


In [14]:
# with open(f"{data_path}South Shore Imported Cars, Inc v. Volkswagen of America, Inc.txt", "rb") as f:
#     raw_data = f.read()
#     result = chardet.detect(raw_data)
#     encoding = result["encoding"]

In [15]:
CONTRACT = "abc123"
with open(f"{data_path}{CONTRACT}.txt", "r", encoding='ISO-8859-1') as f:
    text_str = f.read()


In [16]:
def split_paragraphs_merge_short(text, min_length=200):
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip() != ""]
    chunks = []
    buffer = ""
    
    for para in paragraphs:
        if len(buffer) > 0:
            candidate = buffer + "\n\n" + para
        else:
            candidate = para
        
        if len(candidate) < min_length:
            buffer = candidate
        else:
            chunks.append(candidate)
            buffer = ""
    
    # Add any remaining buffer
    if buffer:
        chunks.append(buffer)
    
    return chunks

In [17]:
output_list = split_paragraphs_merge_short(text_str)

In [18]:
def evaluate(eval_schema, inpt:str, *, system_context: str = "You are an evaluator and a legal scholar, with high attention to detail for errors in contract law. It is VERY IMPORTANT that you spot all errors, especially errors in legal contract reasoning.", model:str = "gpt-5"):
    gpt_response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_context},
            {"role": "user", "content": f"Someone has performed a legal contract analysis and it is now your job to evaluate their output. Their output has five parts. 1: Finds a clause of problematic text in a legal contract. 2: Identifies similar expressions in past contracts/cases retrieved. 3: Explains why those precedents led to disputes or legal issues. 4: Explains why the current clause is problematic by analogy. 5: Proposes a improved wording. \n \n Your job is to evaluate the quality of their problematic wording analysis. \n \n Here is their analysis:\n\n {inpt}"}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "Eval", "schema": eval_schema}
        }
    )
    return gpt_response.choices[0].message.content

eval_schema = {
    "type": "object",
    "properties": {
        "evaluation": {"type": "string"},
        "numeric_score": {"type": "number"}
    }, 
    "required":["evaluation", "numeric_score"]
}


SYSTEM_PROMPT = "You are a very experienced contract-review paralegal for commercial agreements."





In [ ]:
output_list = split_paragraphs_merge_short(text_str)
evaluation_df = pd.DataFrame(columns = ["evaluation", "numeric_score"])
num_problems = int(len(output_list)/5)

for i in range(num_problems):
    row = i*5
    evaluator_output = evaluate(eval_schema=eval_schema, inpt=f"Item 1 is: {output_list[row]}. \n\n Item 2 is: {output_list[row+1]}. \n\n Item 3 is: {output_list[row+2]}. \n\n Item 4 is: {output_list[row+3]}. \n\n Item 5 is: {output_list[row+4]}. ")
    evl = json.loads(evaluator_output)["evaluation"]
    scr = json.loads(evaluator_output)["numeric_score"]
    evaluation_df.loc[len(evaluation_df)] = [evl, scr]

In [ ]:
evaluation_df

In [21]:
num_problems

5